####Spark Aggregate Functions

####Funciones simples de agregacion

In [0]:
%run "../Includes/configuration"

In [0]:
movie_df = spark.read.parquet(f"{silver_folder_path}/movies")

In [0]:
from pyspark.sql.functions import count, countDistinct, sum

In [0]:
movie_df.select(count("*")).show()

In [0]:
movie_df.select(count("year_Release_Date")).show()

In [0]:
movie_df.select(countDistinct("year_Release_Date")).show()

In [0]:
movie_df.select(sum("budget")).display()

In [0]:
display(movie_df.select(sum("budget")))

In [0]:
movie_df.filter("year_Release_Date = 2016") \
        .select(sum("budget"),
                count("movie_Id")) \
        .withColumnRenamed("sum(budget)", "total_budget") \
        .withColumnRenamed("count(movie_Id)", "count_movies") \
        .display()

In [0]:
display(movie_df)

####Group By

In [0]:
movie_df \
    .groupBy("year_Release_Date") \
    .sum("budget") \
    .display()

In [0]:
from pyspark.sql.functions import sum, avg, max, min, count

In [0]:
##Para adicionar mas de una funcion de agregacion hay que agregar la sentencia agg

movie_group_by_df = movie_df \
                    .groupBy("year_Release_Date") \
                        .agg(
                            sum("budget").alias("sum_budget"),
                            avg("budget").alias("avg_budget"),
                            max("budget").alias("max_budget"),
                            min("budget").alias("min_budget"),
                            count("movie_Id").alias("count_movie")
                            )

In [0]:
display(movie_group_by_df)

####Window Function

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, desc, dense_rank

In [0]:
movie_df.select("title", "budget", "year_Release_Date") \
        .filter("year_Release_Date is not null") \
        .withColumn("rank", rank().over(Window.partitionBy("year_Release_Date").orderBy(desc("budget")))) \
        .display()

In [0]:
## la columna budget tiene valores repetidos en el mismo año por lo tanto lo representa con el mismo valor de rank
## para evitar esta ambiguedad se se utilizara la funcion dense_rank y comparemos con rank el
## Se observara que la siguiente enumeracion de los casos ambiguos le otorga el orden correcto sin saltarse posiciones
## Un ejemplo se puede observar en el year_Release_Date = 1960

movie_rank = Window.partitionBy("year_Release_Date").orderBy(desc("budget"))
movie_dense_rank = Window.partitionBy("year_Release_Date").orderBy(desc("budget"))

movie_df.select("title", "budget", "year_Release_Date") \
        .filter("year_Release_Date is not null") \
        .withColumn("rank", rank().over(movie_rank)) \
        .withColumn("dense_rank", dense_rank().over(movie_dense_rank)) \
        .display()